In [1]:
#!/usr/bin/env Rscript

# =============================================================================
# STEP1: Distance & MDS computation (A–D datasets) with isoMDS patch
# =============================================================================

# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------

root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

data_dir_in <- file.path(root, "data", "for_MDS_STEP2")
out_root    <- file.path(root, "data", "calc_core", "01_distance_mds")

datasets <- list(
  A = list(infile = "preprocessed_features_OH.csv",
           subdir = "A_OH_plus_others"),
  B = list(infile = "preprocessed_features_FP.csv",
           subdir = "B_FP_plus_others"),
  C = list(infile = "preprocessed_features_OH_only.csv",
           subdir = "C_OH_only"),
  D = list(infile = "preprocessed_features_FP_only.csv",
           subdir = "D_FP_only")
)

# cmdscale: 最大 250 次元まで保存
mds_k   <- 250

# isoMDS: 「可視化用の低次元埋め込み」として 2 次元だけ求める
iso_dim <- 2

date_str <- format(Sys.Date(), "%Y%m%d")
set.seed(42)

# -----------------------------------------------------------------------------
# Libraries & helpers
# -----------------------------------------------------------------------------

safe_lib <- function(pkg){
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safe_lib("readr")
safe_lib("MASS")

ensure_dir <- function(p){
  if (!dir.exists(p)) {
    dir.create(p, recursive = TRUE, showWarnings = FALSE)
  }
}

read_numeric_matrix <- function(path){
  df <- utils::read.delim(path,
                          header = TRUE, sep = ",",
                          row.names = 1, as.is = TRUE,
                          strip.white = FALSE)
  is_num <- vapply(df, is.numeric, logical(1))
  if (!any(is_num)) stop("No numeric columns in: ", path)
  as.matrix(df[, is_num, drop = FALSE])
}

# -----------------------------------------------------------------------------
# Core computation per dataset
# -----------------------------------------------------------------------------

process_one_dataset <- function(id, meta){
  message("\n======================================================")
  message(sprintf("[INFO] Dataset %s (%s)", id, meta$infile))
  message("======================================================")

  infile  <- file.path(data_dir_in, meta$infile)
  out_dir <- file.path(out_root, meta$subdir)
  ensure_dir(out_dir)

  if (!file.exists(infile)) stop("Input file not found: ", infile)

  # ----- 1. Load data -----
  message("[INFO] Reading data: ", infile)
  numData <- read_numeric_matrix(infile)
  n_samples  <- nrow(numData)
  n_features <- ncol(numData)
  message(sprintf("[INFO] n_samples = %d, n_features = %d", n_samples, n_features))

  # ----- 2. Sample-wise correlation -----
  message("[INFO] Computing sample-wise correlation (pairwise.complete.obs)")
  corMat <- suppressWarnings(stats::cor(t(numData), use = "pairwise.complete.obs"))

  cor_outfile <- file.path(out_dir,
                           sprintf("cor_%s_%s.rds", id, date_str))
  saveRDS(corMat, cor_outfile)
  message("[SAVED] Correlation matrix: ", cor_outfile)

  # ----- 3. Raw distance -----
  message("[INFO] Computing raw distance matrix d_raw = 1 - r_ij")
  d_raw <- 1 - corMat
  diag(d_raw) <- 0

  dist_raw_outfile <- file.path(out_dir,
                                sprintf("distance_raw_%s_%s.rds", id, date_str))
  saveRDS(d_raw, dist_raw_outfile)
  message("[SAVED] Raw distance matrix: ", dist_raw_outfile)

  # ----- 4. Corrected distance -----
  message("[INFO] Computing corrected distance matrix d_corr")

  M     <- n_features
  valid <- !is.na(numData)
  n_eff <- tcrossprod(valid)

  d_corr <- matrix(NA_real_, nrow = n_samples, ncol = n_samples)
  dimnames(d_corr) <- dimnames(corMat)

  ok <- (!is.na(d_raw)) & (n_eff > 1)
  d_corr[ok] <- (M / n_eff[ok]) * d_raw[ok]

  diag(d_corr) <- 0
  d_corr <- pmin(pmax(d_corr, 0), 2)
  d_corr[is.na(d_corr)] <- 2

  dist_corr_outfile <- file.path(out_dir,
                                 sprintf("distance_corr_%s_%s.rds", id, date_str))
  saveRDS(d_corr, dist_corr_outfile)
  message("[SAVED] Corrected distance matrix: ", dist_corr_outfile)

  D_corr <- stats::as.dist(d_corr)

  # ----- 5. Metric MDS (cmdscale) -----
  message("[INFO] Running metric MDS (cmdscale) on corrected distance")

  k_full  <- max(1, min(n_samples - 1, n_samples))
  mds_cmd <- stats::cmdscale(D_corr, k = k_full, eig = TRUE)

  eig_vals    <- mds_cmd$eig
  coords_full <- as.matrix(mds_cmd$points)

  eig_df <- data.frame(
    axis       = seq_along(eig_vals),
    eigenvalue = eig_vals,
    stringsAsFactors = FALSE
  )
  eig_outfile <- file.path(out_dir,
                           sprintf("eigvals_cmdscale_%s_%s.csv", id, date_str))
  readr::write_csv(eig_df, eig_outfile)
  message("[SAVED] cmdscale eigenvalues: ", eig_outfile)

  k_use <- min(mds_k, ncol(coords_full))
  coords_cmd <- coords_full[, seq_len(k_use), drop = FALSE]

  coords_cmd_outfile <- file.path(out_dir,
                                  sprintf("mds_cmdscale_%s_%s.rds", id, date_str))
  saveRDS(coords_cmd, coords_cmd_outfile)
  message(sprintf("[SAVED] cmdscale coordinates (k = %d): %s",
                  k_use, coords_cmd_outfile))

  # ----- 6. Non-metric MDS (isoMDS, low-dim only) -----
  message("[INFO] Running non-metric MDS (isoMDS) with patched distance")

  # パッチ：非対角で 0 以下の距離を eps に置き換え
  D_mat <- as.matrix(D_corr)
  n     <- nrow(D_mat)
  eps   <- 1e-8

  off_diag <- row(D_mat) != col(D_mat)
  D_mat[off_diag & (D_mat <= 0)] <- eps

  D_mat <- (D_mat + t(D_mat)) / 2
  diag(D_mat) <- 0

  D_iso <- stats::as.dist(D_mat)

  # isoMDS は低次元埋め込み用 → 2 次元だけ
  k_iso <- min(iso_dim, k_use)
  message(sprintf("[DEBUG] isoMDS: using k_iso = %d (iso_dim = %d, k_use = %d)",
                  k_iso, iso_dim, k_use))

  init_iso <- coords_cmd[, seq_len(k_iso), drop = FALSE]

  # ここで一度距離の要約を出しておくと、問題があったとき解析しやすい
  message(sprintf("[DEBUG] D_iso summary: min=%.4g, max=%.4g",
                  min(D_iso), max(D_iso)))

  iso_res <- try(
    MASS::isoMDS(D_iso, y = init_iso, k = k_iso, maxit = 50, trace = FALSE),
    silent = TRUE
  )

  if (inherits(iso_res, "try-error")) {
    warning(sprintf("isoMDS failed for dataset %s: %s",
                    id, conditionMessage(attr(iso_res, "condition"))))
  } else {
    coords_iso <- as.matrix(iso_res$points)
    coords_iso_outfile <- file.path(out_dir,
                                    sprintf("mds_isoMDS_%s_%s.rds", id, date_str))
    saveRDS(coords_iso, coords_iso_outfile)
    message("[SAVED] isoMDS coordinates: ", coords_iso_outfile)

    stress_val <- iso_res$stress
    stress_df <- data.frame(
      dataset_id = id,
      date       = date_str,
      n_samples  = n_samples,
      n_features = n_features,
      dim        = k_iso,
      stress     = stress_val,
      stringsAsFactors = FALSE
    )
    stress_outfile <- file.path(out_dir,
                                sprintf("stress_isoMDS_%s_%s.csv", id, date_str))
    readr::write_csv(stress_df, stress_outfile)
    message("[SAVED] isoMDS stress: ", stress_outfile)
  }

  invisible(NULL)
}

# -----------------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------------

ensure_dir(out_root)

for (id in names(datasets)){
  meta <- datasets[[id]]
  process_one_dataset(id, meta)
}

message("\n✅ STEP1 finished. All outputs are under:\n", out_root)


Loading required package: readr

Loading required package: MASS



[INFO] Dataset A (preprocessed_features_OH.csv)


[INFO] Reading data: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv

[INFO] n_samples = 1046, n_features = 394

[INFO] Computing sample-wise correlation (pairwise.complete.obs)

[SAVED] Correlation matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/A_OH_plus_others/cor_A_20251128.rds

[INFO] Computing raw distance matrix d_raw = 1 - r_ij

[SAVED] Raw distance matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/A_OH_plus_others/distance_raw_A_20251128.rds

[INFO] Computing corrected distance matrix d_corr

[SAVED] Correct

[INFO] Computing raw distance matrix d_raw = 1 - r_ij

[SAVED] Raw distance matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/D_FP_only/distance_raw_D_20251128.rds

[INFO] Computing corrected distance matrix d_corr

[SAVED] Corrected distance matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/D_FP_only/distance_corr_D_20251128.rds

[INFO] Running metric MDS (cmdscale) on corrected distance

Warning message in stats::cmdscale(D_corr, k = k_full, eig = TRUE):
"only 501 of the first 1045 eigenvalues are > 0"
[SAVED] cmdscale eigenvalues: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/D_FP_only/eigvals_cmdscale_D_20251128.csv

[SAVED] cmdscale coordinate